# P3 — OOD two-fact composition (positive control)

The one endpoint where an explicit lookup can still beat parametric memory: **out-of-distribution two-hop composition**. A dense model must grok a hard parametric two-hop; a lookup makes both facts co-present (two one-hop lookups + a join).

**Tonight's goal = the kill-gate, not H1.** Confirm the task *clears chance in-distribution* (the check mod-23 iGSM skipped) and read the dense OOD curve. Only if it passes do we spend the seeded 3-arm twin.

Task per row: `What is the {attribute} of {person}'s {mentor|advisor}?` — hop 1 person→bridge, hop 2 bridge→attribute. Populations **P_comp** (composed in training) and **P_held** (atomic facts only, never composed) are disjoint with closed bridge edges.

Runtime: **A100 80GB** recommended (`Runtime → Change runtime type → A100`).

## 1. Setup

In [ ]:
# ---- EDIT THESE TWO LINES ----
CLONE_URL = 'https://github.com/<org>/<repo>.git'   # your repo (private: append a token)
REPO_DIR  = 'Memory-Split'                            # the folder it clones into
# --------------------------------
import os, subprocess
os.chdir('/content')
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', 'p3-composition', CLONE_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
# Scripts self-bootstrap sys.path, but set these too so every entry point is happy:
os.environ['PYTHONPATH'] = os.getcwd()
os.environ['TIKTOKEN_CACHE_DIR'] = os.path.join(os.getcwd(), '.tiktoken_cache')
!pip -q install -r requirements.txt
print('cwd =', os.getcwd())

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
# Optional: persist checkpoints/results to Drive so a disconnect doesn't lose the run
# from google.colab import drive; drive.mount('/content/drive')

## 2. Build the corpus

Defaults are tuned for a one-night positive control: 10k people (8k P_comp / 2k P_held), 600M tokens, atomic:composed ≈ 33:67. Integrity checks (disjoint populations, no held-out leakage, organizer covers every hop) must all pass or the build aborts.

In [ ]:
!python scripts/build_compose.py --out data/compose_v1 \
    --n-entities 10000 --held-frac 0.2 --total-tokens 600_000_000 --n-eval 1000
import json; print(json.dumps(json.load(open('data/compose_v1/report.json'))['checks'], indent=2))

## 3. Positive control — train the dense arm

`d160m` (~162M), ctx 2048, ~0.5M tokens/step, 1B training tokens (~1.6 epochs). ~1–2 h on an A100. Snapshots every 10% give the OOD-vs-step (grokking) curve. Bump `total_tokens`/`micro_batch_size` in the config to use more of the 80GB.

In [ ]:
!python scripts/run_train.py --config configs/compose_dense.yaml --resume auto

## 4. Evaluate + figures

In [ ]:
!python scripts/run_compose_eval.py --run outputs/compose_dense_s0 --data data/compose_v1
from IPython.display import Image, Markdown, display
display(Markdown(open('outputs/compose_dense_s0/compose_eval/summary.md').read()))
for fig in ['ood_vs_step.png', 'accuracy_by_testset.png']:
    p = f'outputs/compose_dense_s0/compose_eval/{fig}'
    import os; display(Image(p)) if os.path.exists(p) else None

## Decision gate
- **PASS** (in-dist ≥ ~30%, single-hop access ≥ ~85%): greenlight the seeded 3-arm twin (dense-closed / dense-open / split).
- **KILL/FIX** (in-dist near chance): the task is mis-scaled — fewer relations/attrs or more composed exposures — *before* spending twin compute.

The dense OOD curve is itself a result: near-floor = the room a lookup could fill; rising = the dense model groks a shortcut (measure, don't assume).

## 5. (Stretch) split arm — needs the token-weighted trainer

The split arm has loss-masked values, so it **must** train with `--trainer v2` (token-weighted gradient accumulation) to avoid the arm-asymmetric bias v1 introduces. Then eval uses the organizer (store ON).

In [ ]:
!python scripts/run_train.py --config configs/compose_split.yaml --trainer v2 --resume auto
!python scripts/run_compose_eval.py --run outputs/compose_split_s0 --data data/compose_v1 --arm split
from IPython.display import Image, Markdown, display
display(Markdown(open('outputs/compose_split_s0/compose_eval/summary.md').read()))
for fig in ['ood_vs_step.png', 'accuracy_by_testset.png', 'per_hop_lookup.png']:
    p = f'outputs/compose_split_s0/compose_eval/{fig}'
    import os; display(Image(p)) if os.path.exists(p) else None